# 🏥 Hospital Operations Data Cleaning & Preprocessing
This notebook demonstrates the end-to-end data cleaning process for our hospital operations dataset. Before applying any transformations, we explicitly check for missing values, invalid entries, and weird artifacts (like the '?' used for nulls in this dataset) to ensure data integrity.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

## Step 1: Splitting the Flat File (Data Normalization)
First, we read the massive 50-column flat file and normalize it into a Star Schema structure (1 Fact Table, 4 Dimension Tables). This allows for efficient SQL modeling later.

In [2]:
# Load the main flat file
df = pd.read_csv('../data/raw/diabetic_data.csv')
print(f"Master Dataset Shape: {df.shape}")

# 1. Patients Dim
patient_cols = ['patient_nbr', 'race', 'gender', 'age', 'weight']
patients_df = df[patient_cols].drop_duplicates(subset=['patient_nbr'])

# 2. Admissions Dim
admission_cols = ['encounter_id', 'patient_nbr', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'payer_code', 'medical_specialty']
admissions_df = df[admission_cols]

# 3. Diagnosis Dim
diag_cols = ['encounter_id', 'patient_nbr', 'diag_1', 'diag_2', 'diag_3', 'number_diagnoses']
diagnoses_df = df[diag_cols]

# 4. Medications Dim
meds_cols = ['encounter_id', 'patient_nbr', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed']
meds_df = df[meds_cols]

# 5. Encounters Fact
fact_cols = ['encounter_id', 'patient_nbr', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'readmitted']
encounters_df = df[fact_cols]

Master Dataset Shape: (101766, 50)


## Step 2: Patients Dimension
### 2.1 Exploratory Data Analysis (EDA)

In [3]:
# Check for missing values encoded as '?'
print("--- PATIENTS TABLE EDA ---")
print("Total rows:", len(patients_df))
print("\nMissing Values ('?'):")
print((patients_df == '?').sum())

print("\nGender Distribution:")
print(patients_df['gender'].value_counts())

--- PATIENTS TABLE EDA ---
Total rows: 71518

Missing Values ('?'):
patient_nbr        0
race            1948
gender             0
age                0
weight         68665
dtype: int64

Gender Distribution:
gender
Female             38025
Male               33490
Unknown/Invalid        3
Name: count, dtype: int64


### 2.2 Data Cleaning
Based on the EDA, 'weight' is 96% missing, 'race' has a few missing values, and 'gender' has invalid entries. Age is categorical and needs to be numeric.

In [4]:
# 1. Drop 'weight' due to >95% missing data
patients_clean = patients_df.drop(columns=['weight'])

# 2. Handle missing 'race'
patients_clean['race'] = patients_clean['race'].replace('?', 'Unknown')

# 3. Drop 'Unknown/Invalid' genders
patients_clean = patients_clean[patients_clean['gender'] != 'Unknown/Invalid']

# 4. Convert 'age' brackets (e.g. '[40-50)') to a numeric midpoint
def get_midpoint(age_str):
    age_str = age_str.replace('[', '').replace(')', '')
    parts = age_str.split('-')
    return (int(parts[0]) + int(parts[1])) / 2

patients_clean['age_midpoint'] = patients_clean['age'].apply(get_midpoint)
patients_clean.drop(columns=['age'], inplace=True)

# Save
patients_clean.to_csv('../data/processed/patients_cleaned.csv', index=False)
print("Patients table cleaned and saved!")

Patients table cleaned and saved!


## Step 3: Admissions Dimension
### 3.1 Exploratory Data Analysis (EDA)

In [5]:
print("--- ADMISSIONS TABLE EDA ---")
print("Total rows:", len(admissions_df))
print("\nMissing Values ('?'):")
print((admissions_df == '?').sum())

--- ADMISSIONS TABLE EDA ---
Total rows: 101766

Missing Values ('?'):
encounter_id                    0
patient_nbr                     0
admission_type_id               0
discharge_disposition_id        0
admission_source_id             0
payer_code                  40256
medical_specialty           49949
dtype: int64


### 3.2 Data Cleaning
'payer_code' and 'medical_specialty' have massive missing values. Instead of dropping columns, we will label them 'Unknown/Missing' to preserve rows for SQL joining.

In [6]:
admissions_clean = admissions_df.copy()

# Fill missing categorical codes
admissions_clean['payer_code'] = admissions_clean['payer_code'].replace('?', 'Unknown')
admissions_clean['medical_specialty'] = admissions_clean['medical_specialty'].replace('?', 'Missing')

# Save
admissions_clean.to_csv('../data/processed/admissions_cleaned.csv', index=False)
print("Admissions table cleaned and saved!")

Admissions table cleaned and saved!


## Step 4: Diagnoses Dimension
### 4.1 Exploratory Data Analysis (EDA)

In [7]:
print("--- DIAGNOSES TABLE EDA ---")
print("Total rows:", len(diagnoses_df))
print("\nMissing Values ('?'):")
print((diagnoses_df == '?').sum())

--- DIAGNOSES TABLE EDA ---
Total rows: 101766

Missing Values ('?'):
encounter_id           0
patient_nbr            0
diag_1                21
diag_2               358
diag_3              1423
number_diagnoses       0
dtype: int64


### 4.2 Data Cleaning
Secondary and tertiary diagnoses are sometimes missing. We replace them with 'None'.

In [8]:
diagnoses_clean = diagnoses_df.copy()

for col in ['diag_1', 'diag_2', 'diag_3']:
    diagnoses_clean[col] = diagnoses_clean[col].replace('?', 'None')

diagnoses_clean.to_csv('../data/processed/diagnoses_cleaned.csv', index=False)
print("Diagnoses table cleaned and saved!")

Diagnoses table cleaned and saved!


## Step 5: Medications Dimension
### 5.1 Exploratory Data Analysis (EDA)

In [9]:
print("--- MEDICATIONS TABLE EDA ---")
print("Unique values in 'examide':", meds_df['examide'].unique())
print("Unique values in 'citoglipton':", meds_df['citoglipton'].unique())
print("\nMissing values (Pandas NaNs) in Lab Results:")
print(meds_df[['max_glu_serum', 'A1Cresult']].isnull().sum())

--- MEDICATIONS TABLE EDA ---
Unique values in 'examide': ['No']
Unique values in 'citoglipton': ['No']

Missing values (Pandas NaNs) in Lab Results:
max_glu_serum    96420
A1Cresult        84748
dtype: int64


### 5.2 Data Cleaning
Lab results have NaNs (not measured). 'examide' and 'citoglipton' have zero variance (all 'No'), so they offer no predictive value.

In [10]:
meds_clean = meds_df.copy()

# Fill unmeasured labs
meds_clean['max_glu_serum'] = meds_clean['max_glu_serum'].fillna('Not Measured')
meds_clean['A1Cresult'] = meds_clean['A1Cresult'].fillna('Not Measured')

# Drop zero-variance columns
meds_clean.drop(columns=['examide', 'citoglipton'], inplace=True)

meds_clean.to_csv('../data/processed/medications_cleaned.csv', index=False)
print("Medications table cleaned and saved!")

Medications table cleaned and saved!


## Step 6: Encounters Fact Table
### 6.1 Exploratory Data Analysis (EDA)

In [11]:
print("--- ENCOUNTERS FACT EDA ---")
print("Missing Values ('?'):")
print((encounters_df == '?').sum())

print("\nReadmitted Categories:")
print(encounters_df['readmitted'].value_counts())

--- ENCOUNTERS FACT EDA ---
Missing Values ('?'):
encounter_id          0
patient_nbr           0
time_in_hospital      0
num_lab_procedures    0
num_procedures        0
num_medications       0
number_outpatient     0
number_emergency      0
number_inpatient      0
readmitted            0
dtype: int64

Readmitted Categories:
readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64


### 6.2 Feature Engineering
The table is perfectly numeric. However, to calculate the Readmission Rate efficiently in SQL, we engineer a binary 'is_readmitted' flag.

In [12]:
encounters_clean = encounters_df.copy()

# Create binary flag
encounters_clean['is_readmitted'] = encounters_clean['readmitted'].apply(lambda x: 0 if x == 'NO' else 1)
encounters_clean.rename(columns={'readmitted': 'readmission_status'}, inplace=True)

encounters_clean.to_csv('../data/processed/encounters_cleaned.csv', index=False)
print("Encounters table cleaned, feature engineered, and saved!")

Encounters table cleaned, feature engineered, and saved!
